In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ipythonx_mvtec_ad_path = kagglehub.dataset_download('ipythonx/mvtec-ad')

print('Data source import complete.')


ModuleNotFoundError: No module named 'kagglehub'

<center><img src="https://raw.githubusercontent.com/openvinotoolkit/anomalib/development/docs/source/images/logos/anomalib-wide-blue.png" alt="Paris" class="center"></center>

<center>💙 A library for benchmarking, developing and deploying deep learning anomaly detection algorithms</center>

---

[Anomalib](https://github.com/openvinotoolkit/anomalib): Anomalib is a deep learning library that aims to collect state-of-the-art anomaly detection algorithms for benchmarking on both public and private datasets. Anomalib provides several ready-to-use implementations of anomaly detection algorithms described in the recent literature, as well as a set of tools that facilitate the development and implementation of custom models. The library has a strong focus on image-based anomaly detection, where the goal of the algorithm is to identify anomalous images, or anomalous pixel regions within images in a dataset.

It supports [`MVTec AD`](https://www.mvtec.com/company/research/datasets/mvtec-ad) (CC BY-NC-SA 4.0) and [`BeanTech`](https://paperswithcode.com/dataset/btad) (CC-BY-SA) for benchmarking and `folder` for custom dataset **training/inference**. In this notebook, we will explore `anomalib` with `MVTec AD` dataset.




## MVTec AD

**MVTec AD** is a dataset for benchmarking anomaly detection methods with a focus on industrial inspection. It contains over **5000** high-resolution images divided into **fifteen** different object and texture categories. Each category comprises a set of defect-free training images and a test set of images with various kinds of defects as well as images without defects. It's uploaded in kaggle platform, [HERE](https://www.kaggle.com/datasets/ipythonx/mvtec-ad). And now, we can use from kaggle environment. For custom dataset, it's a good practice to prepare the dataset according to MVTech format. Normally, we can find the data structure of MVTec-AD for each object as follows:

```yaml
category
  ground_truth
    defect_type_1_mask
    defect_type_2_mask
    ...
  test
    defect_type_1
    defect_type_2
    ...
    good
  train
    good
```

# Installation

In [1]:
!git clone https://github.com/openvinotoolkit/anomalib.git
%cd anomalib
!pip install -e . -q

Cloning into 'anomalib'...
remote: Enumerating objects: 12697, done.
remote: Counting objects: 100% (1132/1132), done.
remote: Compressing objects: 100% (765/765), done.
remote: Total 12697 (delta 665), reused 419 (delta 344), pack-reused 11565 (from 5)
Receiving objects: 100% (12697/12697), 61.05 MiB | 804.00 KiB/s, done.
Resolving deltas: 100% (7339/7339), done.
/home/maria/Documents/projects/anomaly_detection_vehicles/anomalib


# Imports

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import os, pprint, yaml, warnings, math, glob, cv2, random

def warn(*args, **kwargs):
    pass
warnings.warn = warn
warnings.filterwarnings('ignore')

#import anomalib
from pytorch_lightning import Trainer, seed_everything
from anomalib.config import get_configurable_parameters
from anomalib.data import get_datamodule
from anomalib.models import get_model
from anomalib.utils.callbacks import LoadModelCallback, get_callbacks
from anomalib.utils.loggers import get_logger

ModuleNotFoundError: No module named 'anomalib'

In [ ]:
import torch
print(torch.version.cuda)
print(torch.backends.cudnn.version())
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.device(0))
print(torch.cuda.get_device_name(0))

# Model Config Path

Currently, there are **7** anomaly detection models available in `anomalib` library. Namely,

- [Patchcore](https://arxiv.org/pdf/2106.08265.pdf)
- [Padim](https://arxiv.org/pdf/2011.08785.pdf)
- [DFKDE](https://github.com/openvinotoolkit/anomalib/tree/development/anomalib/models/dfkde)
- [DFM](https://arxiv.org/pdf/1909.11786.pdf)
- [CFlow](https://arxiv.org/pdf/2107.12571v1.pdf)
- [Ganomaly](https://arxiv.org/abs/1805.06725)
- [STFPM](https://arxiv.org/pdf/2103.04257.pdf)


Now, let's get their config paths from the respected folders.

In [ ]:
CONFIG_PATHS = '/kaggle/working/anomalib/anomalib/models'
MODEL_CONFIG_PAIRS = {
    'patchcore': f'{CONFIG_PATHS}/patchcore/config.yaml',
    'padim':     f'{CONFIG_PATHS}/padim/config.yaml',
    'cflow':     f'{CONFIG_PATHS}/cflow/config.yaml',
    'dfkde':     f'{CONFIG_PATHS}/dfkde/config.yaml',
    'dfm':       f'{CONFIG_PATHS}/dfm/config.yaml',
    'ganomaly':  f'{CONFIG_PATHS}/ganomaly/config.yaml',
    'stfpm':     f'{CONFIG_PATHS}/stfpm/config.yaml',
}

## Quick Look

In this demonstration, we will choose `PADIM` model from the above config; which is index 1 in the above dictionary. Let's take a quick look of its config file.

In [ ]:
print(open(os.path.join(MODEL_CONFIG_PAIRS['padim']), 'r').read())

# Update Config

In order to train on **MV-Tec** dataset, which is hosted on Kaggle, [HERE](https://www.kaggle.com/datasets/ipythonx/mvtec-ad), we may need to udpate some parameter in the configuration file, for example, `dataset.path`. Also, we may wish to tweak other parameters as well, i.e. `image_size`, `train_batch_size` etc.

In [ ]:
new_update = {
    "path": '/kaggle/input/mvtec-ad',
    'category': 'leather', # there're 15 object in mvtec-ad
    'image_size': 224,
    'train_batch_size':48,
    'seed': 101
}

In the above dictionary, the keys (`path`, `category`, `image_size`, `train_batch_size`) are already exist in the model's config file. We just want to update their values. In the following cell, we write a simple function that will do the job. Note that, in the config file, the `path` key is the nested key of both `dataset` and `project` key. We only need to update the value of `dataset.path` and not `project.path`.

In [ ]:
def update_yaml(old_yaml, new_yaml, new_update):
    with open(old_yaml) as f:
        old = yaml.safe_load(f)

    temp = []
    def set_state(old, key, value):
        if isinstance(old, dict):
            for k, v in old.items():
                if k == 'project':
                    temp.append(k)
                if k == key:
                    if temp and k == 'path':
                        # right now, we don't wanna change `project.path`
                        continue
                    old[k] = value
                elif isinstance(v, dict):
                    set_state(v, key, value)

    for key, value in new_update.items():
        set_state(old, key, value)

    with open(new_yaml, 'w') as f:
        yaml.safe_dump(old, f, default_flow_style=False)

In [ ]:
# let's set a new path location of new config file
new_yaml = CONFIG_PATHS + '/' + list(MODEL_CONFIG_PAIRS.keys())[0] + '_new.yaml'

# run the update yaml method to update desired key's values
update_yaml(MODEL_CONFIG_PAIRS['padim'], new_yaml, new_update)

In [ ]:
with open(new_yaml) as f:new = yaml.safe_load(f)
pprint.pprint(new)

# Prepare Model, Dataloader, Callbacks with `config`

Now, the config file is updated as we want. We can now start model training with it.

In [ ]:
if new['project']['seed'] != 0:
    print(new['project']['seed'])
    seed_everything(new['project']['seed'])

In [ ]:
# It will return the configurable parameters in DictConfig object.
config = get_configurable_parameters(
    model_name=new['model']['name'],
    model_config_path=new_yaml
)

In [ ]:
model      = get_model(config)
logger     = get_logger(config)
callbacks  = get_callbacks(config)
datamodule = get_datamodule(config)

In [ ]:
trainer = Trainer(**config.trainer, logger=logger, callbacks=callbacks)
trainer.fit(model=model, datamodule=datamodule)

In [ ]:
# load best model from checkpoint before evaluating
load_model_callback = LoadModelCallback(
    weights_path=trainer.checkpoint_callback.best_model_path
)
trainer.callbacks.insert(0, load_model_callback)
trainer.test(model=model, datamodule=datamodule)

# Visualization

In [ ]:
RESULT_PATH = os.path.join(new['project']['path'],
                           new['model']['name'],
                           new['dataset']['format'],
                           new['dataset']['category'])
RESULT_PATH

In [ ]:
def vis(paths, n_images, is_random=True, figsize=(16, 16)):
    for i in range(n_images):
        image_name = paths[i]
        if is_random: image_name = random.choice(paths)
        img = cv2.imread(image_name)[:,:,::-1]

        category_type = image_name.split('/')[-4:-3:][0]
        defected_type = image_name.split('/')[-2:-1:][0]

        plt.figure(figsize=figsize)
        plt.imshow(img)
        plt.title(
            f"Category : {category_type} and Defected Type : {defected_type}",
            fontdict={'fontsize': 20, 'fontweight': 'medium'}
        )
        plt.xticks([])
        plt.yticks([])
        plt.tight_layout()
    plt.show()

In [ ]:
for content in os.listdir(RESULT_PATH):
    if content == 'images':
        full_path = glob.glob(os.path.join(RESULT_PATH, content, '**',  '*.png'), recursive=True)
        print(len(full_path))
        print(full_path[0].split('/'))
        print(full_path[0].split('/')[-2:-1:])
        print(full_path[0].split('/')[-4:-3:])

In [ ]:
vis(full_path, 10, is_random=True, figsize=(30, 30))